# Home Depot (HD): Time-Series Stylised Facts, CAPM/APT, ARMA & GARCH

**Author:** Yurui Qiu · Personal project

This notebook:
1. Pulls HD daily/hourly prices and computes log-returns
2. Examines stylised facts (volatility clustering, fat tails, non-normality)
3. Verifies data sources against the Fama-French library
4. Estimates CAPM and a Fama-French five-factor + momentum + oil APT model
5. Fits an ARMA model for the conditional mean and evaluates it out-of-sample
6. Formally tests for ARCH effects and fits a GARCH(1,1) model for the
   conditional variance, then compares ARMA-only vs ARMA+GARCH interval forecasts


## 0. Setup

In [ ]:
import io, zipfile, itertools
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import norm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import bds
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error
import yfinance as yf

sns.set_theme(style="whitegrid")
TICKER = "HD"
MARKET_TICKER = "^GSPC"
TBILL_TICKER = "^IRX"
OIL_TICKER = "CL=F"
TRADING_DAYS = 252

import os
os.makedirs("assets", exist_ok=True)  # works whether run standalone (Colab) or inside the repo


## 1. Load price data and compute log-returns

In [ ]:
def load_prices(ticker, period, interval):
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df

daily = load_prices(TICKER, period="2y", interval="1d")
hourly = load_prices(TICKER, period="60d", interval="1h")

daily["log_return"] = np.log(daily["Close"] / daily["Close"].shift(1))
hourly["log_return"] = np.log(hourly["Close"] / hourly["Close"].shift(1))

daily_lr = daily["log_return"].dropna()
hourly_lr = hourly["log_return"].dropna()
daily_lr.tail(), hourly_lr.tail()


## 2. Stylised facts

### 2a. Return series plots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].plot(daily_lr.index, daily_lr.values, linewidth=0.8)
axes[0].set_title(f"{TICKER} -- Daily Log Returns (last 2 years)")
axes[0].set_xlabel("Date"); axes[0].set_ylabel("Log Return")

axes[1].plot(hourly_lr.index, hourly_lr.values, linewidth=0.8)
axes[1].set_title(f"{TICKER} -- Hourly Log Returns (last 60 days)")
axes[1].set_xlabel("Date"); axes[1].set_ylabel("Log Return")
plt.tight_layout(); plt.savefig("assets/fig1_returns.png", dpi=150); plt.show()


### 2b. Distributional properties

In [ ]:
def hist_with_normal_fit(returns, title, ax):
    mu, sigma = returns.mean(), returns.std(ddof=1)
    ax.hist(returns.values, bins=50, density=True, alpha=0.75)
    x = np.linspace(returns.min(), returns.max(), 400)
    ax.plot(x, norm.pdf(x, loc=mu, scale=sigma), linewidth=2.0, color="darkorange")
    ax.set_title(title); ax.set_xlabel("Log Return"); ax.set_ylabel("Density")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
hist_with_normal_fit(daily_lr, f"{TICKER} Daily -- Histogram vs Normal", axes[0])
hist_with_normal_fit(hourly_lr, f"{TICKER} Hourly -- Histogram vs Normal", axes[1])
plt.tight_layout(); plt.savefig("assets/fig2_histograms.png", dpi=150); plt.show()

def describe_returns(s):
    return pd.Series({
        "Mean": s.mean(),
        "Std Dev": s.std(ddof=1),
        "Skewness": s.skew(),
        "Excess Kurtosis": s.kurtosis(),
    })

stats_table = pd.DataFrame({"Daily": describe_returns(daily_lr), "Hourly": describe_returns(hourly_lr)})
stats_table


## 3. Data matching and verification

Assembles HD returns, S&P 500 returns, the 13-week T-bill rate (`^IRX`), and the
Fama-French five factors + momentum, and checks the sources are consistent with
each other before using them downstream.

In [ ]:
def load_max_history(ticker):
    df = yf.download(ticker, period="max", interval="1d", auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df

hd_hist = load_max_history(TICKER)
mkt_hist = load_max_history(MARKET_TICKER)
tbill_hist = load_max_history(TBILL_TICKER)

hd_hist["ret"] = hd_hist["Close"].pct_change()
mkt_hist["ret"] = mkt_hist["Close"].pct_change()
tbill_hist["rf_annual"] = tbill_hist["Close"] / 100.0

panel = pd.concat([
    hd_hist["ret"].rename("HD_ret"),
    mkt_hist["ret"].rename("MKT_ret"),
    tbill_hist["rf_annual"].rename("Tbill_annual"),
], axis=1).dropna()
panel.head()


In [ ]:
def load_french_factors():
    ff5_url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
    mom_url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip"

    def _read_zip_csv(url, skiprows):
        r = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        return pd.read_csv(z.open(name), skiprows=skiprows)

    ff5 = _read_zip_csv(ff5_url, skiprows=3).rename(columns={"Unnamed: 0": "Date"})
    ff5["Date"] = pd.to_datetime(ff5["Date"], format="%Y%m%d", errors="coerce")
    ff5 = ff5.dropna(subset=["Date"]).set_index("Date").astype(float) / 100.0

    mom = _read_zip_csv(mom_url, skiprows=13).rename(columns={"Unnamed: 0": "Date"})
    mom["Date"] = pd.to_datetime(mom["Date"], format="%Y%m%d", errors="coerce")
    mom = mom.dropna(subset=["Date"]).set_index("Date")
    mom_col = [c for c in mom.columns if c.strip().lower().startswith("mom")][0]
    mom = mom[[mom_col]].astype(float) / 100.0
    mom.columns = ["MOM"]

    return ff5.join(mom, how="left")

ff = load_french_factors()
panel = panel.join(ff, how="inner")
print("Merged panel:", panel.shape, panel.index.min(), "-->", panel.index.max())
panel.head()


### 3a. S&P 500 excess return vs Fama-French market excess return

In [ ]:
panel["SP500_excess"] = panel["MKT_ret"] - panel["RF"]

fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(x=panel["Mkt-RF"], y=panel["SP500_excess"], s=15, alpha=0.6, ax=ax)
sns.regplot(x=panel["Mkt-RF"], y=panel["SP500_excess"], scatter=False, color="red", ax=ax)
ax.set(title="S&P 500 Excess Return vs Fama-French Market Excess Return",
       xlabel="FF Market Excess Return (Mkt-RF)", ylabel="S&P 500 Excess Return")
plt.tight_layout(); plt.savefig("assets/fig3_sp500_vs_ff.png", dpi=150); plt.show()


### 3b. T-bill rate vs Fama-French risk-free rate

In [ ]:
panel["Tbill_daily"] = (1.0 + panel["Tbill_annual"]) ** (1.0 / TRADING_DAYS) - 1.0

fig, ax = plt.subplots(figsize=(7, 6))
tmp = panel[["Tbill_daily", "RF"]].dropna()
sns.scatterplot(x=tmp["Tbill_daily"], y=tmp["RF"], s=15, alpha=0.6, ax=ax)
sns.regplot(x=tmp["Tbill_daily"], y=tmp["RF"], scatter=False, color="red", ax=ax)
ax.set(title="Daily T-bill (^IRX) vs Fama-French Daily RF",
       xlabel="Daily T-bill rate (compounded)", ylabel="FF daily RF")
plt.tight_layout(); plt.savefig("assets/fig4_tbill_vs_ff.png", dpi=150); plt.show()


## 4. CAPM

In [ ]:
capm_df = panel.dropna(subset=["HD_ret", "Mkt-RF", "RF"]).copy()
capm_df["HD_excess"] = capm_df["HD_ret"] - capm_df["RF"]

X = sm.add_constant(capm_df["Mkt-RF"])
y = capm_df["HD_excess"]
capm_model = sm.OLS(y, X).fit()
print(capm_model.summary())

beta_capm = capm_model.params["Mkt-RF"]
alpha_capm = capm_model.params["const"]
print(f"beta = {beta_capm:.3f}, alpha = {alpha_capm:.5f}")


In [ ]:
# Replication portfolio: RF + beta * (Mkt - RF), compared with HD directly
capm_df["rep_portfolio"] = capm_df["RF"] + beta_capm * capm_df["Mkt-RF"]

def perf_stats(returns, rf):
    mean_r, var_r = returns.mean(), returns.var()
    sharpe = (mean_r - rf.mean()) / returns.std()
    return mean_r, var_r, sharpe

perf_table = pd.DataFrame({
    "Replication Portfolio": perf_stats(capm_df["rep_portfolio"], capm_df["RF"]),
    "Home Depot (HD)": perf_stats(capm_df["HD_ret"], capm_df["RF"]),
}, index=["Mean E[R]", "Variance", "Sharpe Ratio"])
perf_table


## 5. Arbitrage Pricing Theory (Fama-French five-factor + momentum + oil)

In [ ]:
apt_df = panel.dropna(subset=["HD_ret", "RF", "Mkt-RF", "SMB", "HML", "RMW", "CMA"]).copy()
apt_df["HD_excess"] = apt_df["HD_ret"] - apt_df["RF"]

X5 = sm.add_constant(apt_df[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]])
ff5_model = sm.OLS(apt_df["HD_excess"], X5).fit()
print(ff5_model.summary())


In [ ]:
apt_mom_df = panel.dropna(subset=["HD_ret", "RF", "Mkt-RF", "SMB", "HML", "RMW", "CMA", "MOM"]).copy()
apt_mom_df["HD_excess"] = apt_mom_df["HD_ret"] - apt_mom_df["RF"]

X6 = sm.add_constant(apt_mom_df[["Mkt-RF", "SMB", "HML", "RMW", "CMA", "MOM"]])
ff6_model = sm.OLS(apt_mom_df["HD_excess"], X6).fit()
print(f"FF5 adj. R^2 = {ff5_model.rsquared_adj:.3f} | FF5+MOM adj. R^2 = {ff6_model.rsquared_adj:.3f}")
print(ff6_model.summary())


In [ ]:
oil = load_max_history(OIL_TICKER)[["Close"]].rename(columns={"Close": "Oil_close"})
oil["Oil_ret"] = oil["Oil_close"].pct_change()

apt_oil_df = apt_mom_df.join(oil["Oil_ret"], how="inner").dropna(subset=["Oil_ret"])
X7 = sm.add_constant(apt_oil_df[["Mkt-RF", "SMB", "HML", "RMW", "CMA", "MOM", "Oil_ret"]])
ff7_model = sm.OLS(apt_oil_df["HD_excess"], X7).fit()

print(f"FF5+MOM adj. R^2 = {ff6_model.rsquared_adj:.3f} | +Oil adj. R^2 = {ff7_model.rsquared_adj:.3f}")
print(ff7_model.summary())


In [ ]:
# Multicollinearity check
X_vif = X7.drop(columns="const")
vif_table = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
vif_table


## 6. ARMA model for the conditional mean

### 6a. ACF / PACF of returns and squared returns

In [ ]:
returns = apt_mom_df["HD_ret"].dropna()

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
plot_acf(returns, lags=10, ax=axes[0, 0]); axes[0, 0].set_title("ACF of Returns")
plot_pacf(returns, lags=10, ax=axes[0, 1]); axes[0, 1].set_title("PACF of Returns")
plot_acf(returns**2, lags=10, ax=axes[1, 0]); axes[1, 0].set_title("ACF of Squared Returns")
plot_pacf(returns**2, lags=10, ax=axes[1, 1]); axes[1, 1].set_title("PACF of Squared Returns")
plt.tight_layout(); plt.savefig("assets/fig5_acf_pacf.png", dpi=150); plt.show()


### 6b. Grid search over low-order ARMA(p,q) specifications

In [ ]:
def search_arma_orders(series, max_order=2):
    results = []
    for p, q in itertools.product(range(max_order + 1), repeat=2):
        try:
            fit = ARIMA(series, order=(p, 0, q), trend="c",
                        enforce_stationarity=False, enforce_invertibility=False).fit()
            results.append({"order": (p, q), "aic": fit.aic, "bic": fit.bic, "model": fit})
        except Exception:
            continue
    return pd.DataFrame(results).sort_values("aic")

arma_grid = search_arma_orders(returns)
arma_grid[["order", "aic", "bic"]]


In [ ]:
best_row = arma_grid.iloc[0]
best_order = best_row["order"]
best_model = best_row["model"]
print(f"Selected ARMA{best_order} by AIC")
print(best_model.summary())


### 6c. Residual diagnostics: Ljung-Box and BDS

In [ ]:
resid = best_model.resid
std_resid = (resid - resid.mean()) / resid.std(ddof=0)

for lag in (10, 20, 30):
    p_val = float(acorr_ljungbox(resid, lags=[lag], return_df=True)["lb_pvalue"])
    print(f"Ljung-Box p(lag={lag}) = {p_val:.4g}")

bds_stat, bds_pval = bds(std_resid, max_dim=6)
print("\nBDS test (non-linear dependence in standardized residuals):")
print(pd.DataFrame({"m": range(2, 7), "BDS stat": bds_stat, "p-value": bds_pval}))


### 6d. Long-run forecast

In [ ]:
const, phi = best_model.params.get("const", np.nan), best_model.params.get("ar.L1", 0.0)
mu_inf = const / (1 - phi) if abs(phi) < 1 else np.nan
print(f"Analytical long-run mean = {mu_inf:.6f}")

fcst = best_model.get_forecast(steps=1000)
pred_mean, pred_ci = fcst.predicted_mean, fcst.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pred_mean.values, label="Dynamic forecast (mean)")
ax.axhline(mu_inf, linestyle="--", color="black", label="Analytical long-run mean")
ax.fill_between(range(len(pred_mean)), pred_ci.iloc[:, 0], pred_ci.iloc[:, 1], alpha=0.25, label="95% CI")
ax.set(title="1000-step Dynamic Forecast vs Long-run Mean", xlabel="Forecast step", ylabel="Return")
ax.legend(); plt.tight_layout(); plt.savefig("assets/fig6_longrun_forecast.png", dpi=150); plt.show()


### 6e-f. Out-of-sample evaluation vs a white-noise benchmark

In [ ]:
TEST_SIZE = 100
train, test = returns.iloc[:-TEST_SIZE], returns.iloc[-TEST_SIZE:]

arma_oos = ARIMA(train, order=(best_order[0], 0, best_order[1]), trend="c").fit()
fcst_oos = arma_oos.get_forecast(steps=TEST_SIZE)
pred = pd.Series(fcst_oos.predicted_mean.values, index=test.index)

rmse_arma = np.sqrt(mean_squared_error(test, pred))
rmse_white = np.sqrt(mean_squared_error(test, pd.Series(train.mean(), index=test.index)))

print(f"ARMA{best_order} RMSE      = {rmse_arma:.6f}")
print(f"White-noise RMSE          = {rmse_white:.6f}")
print("ARMA beats white noise" if rmse_arma < rmse_white else "ARMA does not beat white noise")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train.index, train, linewidth=0.6, label="Training data")
ax.plot(test.index, test, linewidth=1.2, label="Actual (test)")
ax.plot(pred.index, pred, color="darkorange", label="ARMA forecast")
ax.set(title=f"ARMA{best_order} Out-of-Sample Forecast vs Actual", xlabel="Date", ylabel="Return")
ax.legend(); plt.tight_layout(); plt.savefig("assets/fig7_oos_forecast.png", dpi=150); plt.show()


## 7. Extension: formal ARCH test and GARCH(1,1) volatility model

Section 6's residual diagnostics showed persistent autocorrelation in squared
residuals and a BDS test that strongly rejects i.i.d. residuals -- both signs
of conditional heteroskedasticity that an ARMA mean model cannot capture on its
own. This section follows up on that: it formally tests for ARCH effects
(rather than relying on visual inspection of the ACF), fits a GARCH(1,1) on top
of the chosen ARMA mean model, and compares the resulting interval forecasts
with the ARMA-only intervals used in Section 6.

### 7a. Engle's ARCH-LM test

In [ ]:
arch_lm_stat, arch_lm_pval, _, _ = het_arch(resid, nlags=10)
print(f"Engle ARCH-LM statistic = {arch_lm_stat:.2f}, p-value = {arch_lm_pval:.4g}")
print("Reject homoskedasticity -> conditional variance should be modelled explicitly."
      if arch_lm_pval < 0.05 else "No strong evidence of ARCH effects at the 5% level.")


### 7b. Fit GARCH(1,1) on the ARMA residuals

In [ ]:
# pip install arch
from arch import arch_model

resid_pct = resid * 100  # arch_model scales more stably on percentage returns
garch_fit = arch_model(resid_pct, mean="Zero", vol="GARCH", p=1, q=1, dist="t").fit(disp="off")
print(garch_fit.summary())


### 7c. Conditional volatility and ARMA vs ARMA-GARCH interval forecasts

In [ ]:
cond_vol = garch_fit.conditional_volatility / 100  # back to return scale

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(resid.index, cond_vol.values, linewidth=0.8, color="firebrick")
ax.set(title="GARCH(1,1) Conditional Volatility of ARMA Residuals",
       xlabel="Date", ylabel="Conditional Std. Dev.")
plt.tight_layout(); plt.savefig("assets/fig8_garch_volatility.png", dpi=150); plt.show()


In [ ]:
# Out-of-sample: compare a constant (ARMA-implied) forecast interval with the
# time-varying GARCH interval over the same TEST_SIZE window used in 6e-f.
garch_oos = arch_model(train.rename("HD_ret") * 100, mean="Zero", vol="GARCH", p=1, q=1, dist="t").fit(disp="off")
garch_forecast = garch_oos.forecast(horizon=TEST_SIZE, reindex=False)
garch_sigma = np.sqrt(garch_forecast.variance.values[-1]) / 100  # per-step conditional std dev

arma_sigma_const = np.sqrt(arma_oos.params["sigma2"])  # ARMA's single constant variance

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test.index, test.values, label="Actual (test)", linewidth=1.2)
ax.plot(pred.index, pred.values, color="darkorange", label="ARMA forecast mean")
ax.fill_between(test.index, pred.values - 1.96*arma_sigma_const, pred.values + 1.96*arma_sigma_const,
                alpha=0.15, color="gray", label="ARMA-only 95% CI (constant width)")
ax.fill_between(test.index, pred.values - 1.96*garch_sigma, pred.values + 1.96*garch_sigma,
                alpha=0.25, color="firebrick", label="ARMA+GARCH 95% CI (time-varying width)")
ax.set(title="ARMA-only vs ARMA+GARCH Forecast Intervals", xlabel="Date", ylabel="Return")
ax.legend(); plt.tight_layout(); plt.savefig("assets/fig9_arma_vs_garch_intervals.png", dpi=150); plt.show()

print(f"Mean ARMA-only interval half-width   = {1.96*arma_sigma_const:.5f} (constant)")
print(f"Mean ARMA+GARCH interval half-width  = {(1.96*garch_sigma).mean():.5f} (time-varying, "
      f"range {(1.96*garch_sigma).min():.5f}-{(1.96*garch_sigma).max():.5f})")


### 7d. Interpretation

If the GARCH-based intervals widen around volatile periods and narrow during
calm periods (rather than the constant-width ARMA-only band), that confirms the
conclusion from Sections 6 and 7a: the conditional **mean** of HD's returns is
close to white noise (ARMA barely beats it), but the conditional **variance**
is genuinely time-varying and worth modelling separately for risk-management
purposes such as Value-at-Risk.